---
# 1 · Konfigurasi Dataset

In [1]:
from pathlib import Path

# ============================================================
# EDIT BASE PATH dataset Anda
# ============================================================
BASE = Path(r"E:\competition\Datathon\dataset")

# Berapa file di-sample untuk inspeksi detail (untuk speed)
SAMPLE_PER_FOLDER = 1   

# ============================================================
# DAFTAR DATASET — sesuaikan path-nya
# ============================================================
DATASETS = [
    {
        "name": "MUMBAI",
        "image_files": [BASE / "India" / "Mumbai" / "S2" / "training" / "Mumbai.tif"],
        "label_files": [BASE / "India" / "Mumbai" / "S2" / "training" / "Mumbai_ground_truth.tif"],
    },
    {
        "name": "NAIROBI_KIBERA",
        "image_files": [BASE / "Nairobi" / "Kibera" / "S2" / "training" / "Kibera.tif"],
        "label_files": [BASE / "Nairobi" / "Kibera" / "S2" / "training" / "Kibera_ground_truth.tif"],
    },
    {
        "name": "NAIROBI_KIANDA",
        "image_files": [BASE / "Nairobi" / "Kianda" / "S2" / "training" / "Kianda.tif"],
        "label_files": [BASE / "Nairobi" / "Kianda" / "S2" / "training" / "Kianda_ground_truth.tif"],
    },
    {
        "name": "NAIROBI_LOWER",
        "image_files": [BASE / "Nairobi" / "Lower" / "S2" / "training" / "Lower.tif"],
        "label_files": [BASE / "Nairobi" / "Lower" / "S2" / "training" / "Lower_ground_truth.tif"],
    },
    {
        "name": "INDONESIA",
        "image_dirs": [
            BASE / "Indonesia" / "images" / "slum",
            BASE / "Indonesia" / "images" / "non_slum",
        ],
        "geojson_dirs": [
            BASE / "Indonesia" / "labels",
        ],
    },
]

print(f"BASE path: {BASE}")
print(f"BASE exists: {BASE.exists()}")
print(f"Datasets to inspect: {len(DATASETS)}")
for d in DATASETS:
    print(f"  - {d['name']}")

BASE path: E:\competition\Datathon\dataset
BASE exists: True
Datasets to inspect: 5
  - MUMBAI
  - NAIROBI_KIBERA
  - NAIROBI_KIANDA
  - NAIROBI_LOWER
  - INDONESIA


---
# 2 · Helper Functions

In [ ]:
import rasterio
import numpy as np
import geopandas as gpd
import warnings
warnings.filterwarnings("ignore")

def list_tifs(folder):
    if not folder.exists():
        return []
    return sorted(list(folder.glob("*.tif")) + list(folder.glob("*.tiff")))

def list_geojsons(folder):
    if not folder.exists():
        return []
    return sorted(list(folder.glob("*.geojson")) + list(folder.glob("*.json")))


def inspect_tif_basic(path, label="Image"):
    """Print info dasar TIF: resolusi, band, CRS, dtype, shape."""
    print(f"File sampel ({label}): {path}")
    try:
        with rasterio.open(path) as src:
            print(f"Resolusi      : ({abs(src.transform[0])}, {abs(src.transform[4])}) "
                  f"{'meter/pixel' if src.crs and src.crs.is_projected else 'derajat/pixel (lat/lon)'}")
            print(f"Jumlah band   : {src.count}")
            print(f"Deskripsi band: {src.descriptions}")
            print(f"CRS           : {src.crs}")
            print(f"Dtype         : {src.dtypes}")
            print(f"Shape         : {src.height} x {src.width} pixel")
            print(f"Bounds        : {src.bounds}")
            print(f"NoData        : {src.nodata}")
            return src.count, src.dtypes[0], (src.height, src.width)
    except Exception as e:
        print(f"ERROR: {e}")
        return None, None, None


def inspect_tif_values(path, max_bands=6):
    """Print value range per band (sampai max_bands)."""
    try:
        with rasterio.open(path) as src:
            print(f"Per-band stats (sample {min(src.count, max_bands)} band pertama):")
            print(f"  {'Band':<6s} {'min':>12s} {'p1':>12s} {'mean':>12s} {'p99':>12s} {'max':>12s}")
            for b in range(min(src.count, max_bands)):
                arr = src.read(b + 1)
                if src.nodata is not None:
                    arr = arr[arr != src.nodata]
                else:
                    arr = arr[~np.isnan(arr)] if arr.dtype.kind == "f" else arr.flatten()
                if arr.size == 0:
                    print(f"  [{b+1}] (no valid pixels)")
                    continue
                print(f"  [{b+1:<3d}] {float(arr.min()):>12.4g} {float(np.percentile(arr,1)):>12.4g} "
                      f"{float(arr.mean()):>12.4g} {float(np.percentile(arr,99)):>12.4g} "
                      f"{float(arr.max()):>12.4g}")
    except Exception as e:
        print(f"ERROR baca values: {e}")


def inspect_label_tif(path):
    """Khusus label TIF: print unique values + distribusi."""
    print(f"Label file    : {path}")
    try:
        with rasterio.open(path) as src:
            arr = src.read()
            print(f"Label shape   : {arr.shape}")
            print(f"Label dtype   : {src.dtypes[0]}")
            unique, counts = np.unique(arr, return_counts=True)
            total = arr.size
            print(f"Nilai unik    : {unique.tolist()}")
            print(f"Distribusi    :")
            for v, c in zip(unique, counts):
                pct = c / total * 100
                print(f"  value={v}: {c:>12,d} pixel ({pct:>6.2f}%)")
    except Exception as e:
        print(f"ERROR: {e}")


def inspect_geojson(path):
    """Print info GeoJSON: features, columns, CRS, geom types."""
    print(f"File sampel   : {path}")
    try:
        gdf = gpd.read_file(path)
        print(f"Jumlah fitur  : {len(gdf)}")
        print(f"Kolom         : {list(gdf.columns)}")
        print(f"CRS           : {gdf.crs}")
        print(f"Tipe geometri : {gdf.geom_type.unique().tolist()}")
        print(f"Bounds        : {tuple(gdf.total_bounds)}")
        n_valid = int(gdf.geometry.is_valid.sum())
        n_empty = int(gdf.geometry.is_empty.sum())
        print(f"Valid/Empty   : {n_valid} valid, {n_empty} empty, {len(gdf)-n_valid} invalid")
        if gdf.crs is not None:
            try:
                gdf_proj = gdf.to_crs(gdf.estimate_utm_crs()) if not gdf.crs.is_projected else gdf
                area_km2 = gdf_proj.geometry.area.sum() / 1e6
                print(f"Total area    : {area_km2:.3f} km²")
            except Exception:
                pass
        print(f"Preview data:")
        print(gdf.head(3).to_string())
    except Exception as e:
        print(f"ERROR: {e}")


print("Helper functions loaded.")

Helper functions loaded.


---
# 3 · Inspect Per Dataset

In [3]:
def inspect_dataset(ds):
    name = ds["name"]
    print()
    print("=" * 70)
    print(f"=== {name} ===")
    print("=" * 70)

    # --- IMAGE FILES (single files) ---
    if "image_files" in ds:
        for img_path in ds["image_files"]:
            if not img_path.exists():
                print(f"❌ File tidak ditemukan: {img_path}")
                continue
            print()
            n_bands, dtype, shape = inspect_tif_basic(img_path, "Image")
            inspect_tif_values(img_path, max_bands=6)

    # --- IMAGE DIRS (banyak file di folder) ---
    if "image_dirs" in ds:
        for img_dir in ds["image_dirs"]:
            tifs = list_tifs(img_dir)
            print()
            print(f"[INFO] Ditemukan {len(tifs)} file .tif di {img_dir}")
            if not tifs:
                continue
            for sample in tifs[:SAMPLE_PER_FOLDER]:
                inspect_tif_basic(sample, "Image")
                inspect_tif_values(sample, max_bands=6)

    # --- LABEL TIF (ground truth) ---
    if "label_files" in ds:
        for lbl_path in ds["label_files"]:
            if not lbl_path.exists():
                print(f"❌ Label tidak ditemukan: {lbl_path}")
                continue
            print()
            inspect_label_tif(lbl_path)

    # --- GEOJSON DIRS ---
    if "geojson_dirs" in ds:
        for geo_dir in ds["geojson_dirs"]:
            geos = list_geojsons(geo_dir)
            print()
            print(f"[INFO] Ditemukan {len(geos)} file .geojson di {geo_dir}")
            if not geos:
                continue
            for sample in geos[:SAMPLE_PER_FOLDER]:
                print()
                print(f"=== {name} (labels GeoJSON) ===")
                inspect_geojson(sample)


# Run inspeksi semua dataset
for ds in DATASETS:
    inspect_dataset(ds)

print()
print("=" * 70)
print("INSPEKSI SELESAI")
print("=" * 70)


=== MUMBAI ===

File sampel (Image): E:\competition\Datathon\dataset\India\Mumbai\S2\training\Mumbai.tif
Resolusi      : (10.0, 10.0) meter/pixel
Jumlah band   : 10
Deskripsi band: (None, None, None, None, None, None, None, None, None, None)
CRS           : EPSG:32643
Dtype         : ('float32', 'float32', 'float32', 'float32', 'float32', 'float32', 'float32', 'float32', 'float32', 'float32')
Shape         : 3927 x 1993 pixel
Bounds        : BoundingBox(left=266800.0, bottom=2092150.0, right=286730.0, top=2131420.0)
NoData        : None
Per-band stats (sample 6 band pertama):
  Band            min           p1         mean          p99          max
  [1  ]            1          890         1235         1747    1.538e+04
  [2  ]           67          713         1058         1665    1.506e+04
  [3  ]          323          447        861.8         1781    1.996e+04
  [4  ]          372          532        904.4         1675    1.209e+04
  [5  ]          319          373         1190    

---
# 4 · Tabel Perbandingan Cross-Dataset

In [4]:
import pandas as pd

def quick_summary(ds):
    """Return dict ringkasan 1 dataset."""
    summary = {"dataset": ds["name"]}
    sample_path = None
    if "image_files" in ds:
        for p in ds["image_files"]:
            if p.exists():
                sample_path = p
                break
    if sample_path is None and "image_dirs" in ds:
        for d in ds["image_dirs"]:
            tifs = list_tifs(d)
            if tifs:
                sample_path = tifs[0]
                summary["n_files"] = len(tifs)
                break
    if sample_path is None:
        return summary

    try:
        with rasterio.open(sample_path) as src:
            summary["bands"]  = src.count
            summary["dtype"]  = src.dtypes[0]
            summary["crs"]    = str(src.crs)
            summary["res"]    = round(abs(src.transform[0]), 6)
            summary["shape"]  = f"{src.height}x{src.width}"
            arr = src.read(1)
            if src.nodata is not None:
                arr = arr[arr != src.nodata]
            summary["p1"]     = round(float(np.percentile(arr, 1)), 4)
            summary["p99"]    = round(float(np.percentile(arr, 99)), 4)
    except Exception as e:
        summary["error"] = str(e)

    # Detect scale
    p99 = summary.get("p99", 0)
    if p99 <= 1.5:
        summary["scale"] = "0-1 (reflectance)"
    elif 1000 < p99 <= 20000:
        summary["scale"] = "0-10000 (raw DN)"
    elif p99 <= 255:
        summary["scale"] = "0-255 (8-bit)"
    elif p99 > 20000:
        summary["scale"] = "uint16 raw"
    else:
        summary["scale"] = "?"
    return summary

summaries = [quick_summary(ds) for ds in DATASETS]
df = pd.DataFrame(summaries)
print("\n" + "=" * 80)
print("RINGKASAN CROSS-DATASET")
print("=" * 80)
print(df.to_string(index=False))


RINGKASAN CROSS-DATASET
       dataset  bands   dtype        crs      res     shape       p1     p99             scale  n_files
        MUMBAI     10 float32 EPSG:32643 10.00000 3927x1993 890.0000 1747.00  0-10000 (raw DN)      NaN
NAIROBI_KIBERA     10 float32 EPSG:32737 16.00000   118x256 870.0000 1953.93  0-10000 (raw DN)      NaN
NAIROBI_KIANDA     10 float32 EPSG:32737 16.00000     42x34 890.5400 1866.46  0-10000 (raw DN)      NaN
 NAIROBI_LOWER     10 float32 EPSG:32737 10.00000    46x567   0.0000 1970.19  0-10000 (raw DN)      NaN
     INDONESIA      6 float64  EPSG:4326  0.00009   173x154   0.0486    0.30 0-1 (reflectance)    140.0


---
# 5 · Diagnosis Kompatibilitas dengan Prithvi-EO-2.0

In [ ]:
print("=" * 80)
print("DIAGNOSIS KOMPATIBILITAS PRITHVI 6-BAND HLS")
print("=" * 80)
print("Target: [B2, B3, B4, B8A, B11, B12] (BLUE, GREEN, RED, NIR_NARROW, SWIR_1, SWIR_2)")
print()

for s in summaries:
    name = s["dataset"]
    bands = s.get("bands")
    scale = s.get("scale", "?")
    crs   = s.get("crs", "?")
    print(f"--- {name} ---")
    if bands is None:
        print(f"tidak bisa baca file. Cek path/permission.")
        continue
    # Band count diagnosis
    if bands == 6:
        print(f"6 band — match Prithvi (asumsi urutan B2,B3,B4,B8A,B11,B12)")
    elif bands == 4:
        print(f"4 band — kemungkinan [B2,B3,B4,B8] (BGRN tanpa SWIR & B8A)")
        print(f"Tidak bisa langsung pakai Prithvi 6-band tanpa SWIR.")
        print(f"Opsi: (a) re-export dari source dengan 6+ band, atau")
        print(f"(b) duplicate channel / pad zeros (kualitas turun signifikan)")
    elif bands == 10:
        print(f" 10 band — kemungkinan Sentinel-2 standar")
        print(f"[B2,B3,B4,B5,B6,B7,B8,B8A,B11,B12] → ambil index [0,1,2,7,8,9]")
    elif bands == 19:
        print(f"19 band — kemungkinan FDL extended (sentinel + indices + label)")
        print(f"Perlu cek deskripsi band atau dokumentasi source untuk identifikasi")
    else:
        print(f"{bands} band — non-standard, perlu cek manual mapping")
    # Scale diagnosis
    print(f"  Skala       : {scale}")
    if scale == "0-255 (8-bit)":
        print(f"TERLALU LOSSY untuk Prithvi — re-export dari source")
    # CRS
    if "EPSG:4326" in crs:
        print(f"  CRS         : {crs} (lat/lon — perlu reproject ke UTM saat preprocessing)")
    elif "EPSG:" in crs:
        print(f"  CRS         : {crs} (projected — bagus)")
    else:
        print(f"  CRS         : {crs}")
    print()

DIAGNOSIS KOMPATIBILITAS PRITHVI 6-BAND HLS
Target: [B2, B3, B4, B8A, B11, B12] (BLUE, GREEN, RED, NIR_NARROW, SWIR_1, SWIR_2)

--- MUMBAI ---
 10 band — kemungkinan Sentinel-2 standar
[B2,B3,B4,B5,B6,B7,B8,B8A,B11,B12] → ambil index [0,1,2,7,8,9]
  Skala       : 0-10000 (raw DN)
  CRS         : EPSG:32643 (projected — bagus)

--- NAIROBI_KIBERA ---
 10 band — kemungkinan Sentinel-2 standar
[B2,B3,B4,B5,B6,B7,B8,B8A,B11,B12] → ambil index [0,1,2,7,8,9]
  Skala       : 0-10000 (raw DN)
  CRS         : EPSG:32737 (projected — bagus)

--- NAIROBI_KIANDA ---
 10 band — kemungkinan Sentinel-2 standar
[B2,B3,B4,B5,B6,B7,B8,B8A,B11,B12] → ambil index [0,1,2,7,8,9]
  Skala       : 0-10000 (raw DN)
  CRS         : EPSG:32737 (projected — bagus)

--- NAIROBI_LOWER ---
 10 band — kemungkinan Sentinel-2 standar
[B2,B3,B4,B5,B6,B7,B8,B8A,B11,B12] → ambil index [0,1,2,7,8,9]
  Skala       : 0-10000 (raw DN)
  CRS         : EPSG:32737 (projected — bagus)

--- INDONESIA ---
6 band — match Prithvi (asu